# Notebook 8 — Counter-example augmented fine-tuning on Mohler (5 folds)  ·  GPU: T4

Identical to Notebook 3 (same folds, prompt, hyper-parameters, seeds, validation-NLL checkpoint rule) except that each fold's training set is **augmented with synthetic wrong answers built only from that fold's training answers** (`perturb.make_counterexamples`):

- sources: training answers with gold ≥ 8 and no negation words; synthetic examples = 20% of the training set, split equally over
- `negated` (rule-based *not* insertion) → target 0 · `keyword_salad` (shuffled reference content words) → target 0 · `swapped` (a correct answer to a *different training* question) → target 0.

Validation (checkpoint selection) uses only the real validation answers. Test questions are never touched.

**Resumable inside a fold:** the full training state is saved to `ckpt/` at every evaluation (twice per epoch), so a disconnect loses at most ~10 min. Re-run all cells to continue. ~40 min per fold → about 2 sessions.

**Before running:** upload the updated `llm_utils.py` and `perturb.py`. T4 runtime.

In [1]:
import time
SESSION_START = time.time()
BUDGET_H = 2.3      # do not START a new fold after this (an interrupted fold resumes anyway)
from google.colab import drive
drive.mount('/content/drive')
import os, sys, shutil
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)
for d in ['models', 'ckpt']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
!pip -q install -U bitsandbytes accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 48.7 MB/s eta 0:00:00


In [3]:
import json, gc, platform
import numpy as np, pandas as pd, torch, transformers, bitsandbytes, peft, nltk
for p in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(p, quiet=True)
from asag_utils import *
from llm_utils import *
from perturb import make_counterexamples, AUG_RATIO
versions = {'torch': torch.__version__, 'transformers': transformers.__version__, 'bitsandbytes': bitsandbytes.__version__,
            'peft': peft.__version__, 'gpu': torch.cuda.get_device_name(0)}
mo = pd.read_csv(f'{ROOT}/data/mohler_prepared.csv')
prompts = make_prompts(mo, 'mohler')
TAG = 'llama3_ftaug'
print('Folds finished:', [k for k in range(N_FOLDS) if os.path.exists(f'{ROOT}/results/mohler_{TAG}_f{k}.csv')])

Folds finished: []


## 1. Train and predict, fold by fold

In [ ]:
lp_cols = [f'lp_{s}' for s in SCORE_TARGETS]
for k in range(N_FOLDS):
    out = f'{ROOT}/results/mohler_{TAG}_f{k}.csv'
    if os.path.exists(out):
        continue
    ck = f'{ROOT}/ckpt/mohler_{TAG}_f{k}'
    if (time.time() - SESSION_START) / 3600 > BUDGET_H and not os.path.exists(f'{ck}/state.pt'):
        print(f'Budget reached: stopping before fold {k}. Re-run all cells in a new session.'); break
    print(f'==================== fold {k} ====================')
    t_fold = time.time()
    model, tok = load_hf_4bit(BASE_MODEL)
    cand, _ = candidate_ids(tok, SCORE_TARGETS)
    model, n_trainable = add_lora(model, seed=SEED + k)
    split = mo[f'split_f{k}']
    tr, va, te = mo[split == 'train'], mo[split == 'val'], mo[split == 'test']
    aug = make_counterexamples(tr, 'mohler', seed=SEED + k)
    assert not set(aug.group) - set(tr.group), 'augmentation must come from training questions only'
    ex = build_examples(tok, [prompts[i] for i in tr.index] + make_prompts(aug, 'mohler'),
                        tr.y_int.astype(str).tolist() + aug.target.astype(str).tolist())
    ys = list(tr.y_int) + list(aug.target)
    assert all(len(t) == 2 and t[0] == cand[y] for (_, t), y in zip(ex, ys)), 'target tokenisation mismatch'
    print(f'train {len(tr)} + synthetic {len(aug)} {aug.aug_type.value_counts().to_dict()}  val {len(va)}  test {len(te)}')

    val_texts, vy_int, vy = [prompts[i] for i in va.index], va.y_int.values, va.y.values
    def evaluate():
        lp = candidate_logprobs(model, tok, val_texts, cand, batch_size=8)
        e, _, m = score_from_logprobs(lp)
        return {'val_nll': float(-lp[np.arange(len(vy_int)), vy_int].mean()),
                'val_pearson': float(np.corrcoef(e, vy)[0, 1]), 'val_mae': float(np.abs(e - vy).mean()), 'val_mass': float(m.mean())}

    torch.cuda.reset_peak_memory_stats()
    hist, best = train_lora(model, tok, ex, evaluate, seed=SEED + k, ckpt_path=ck)
    model.save_pretrained(f'{ROOT}/models/mohler_{TAG}_f{k}')
    rows = pd.concat([va, te])
    lp = candidate_logprobs(model, tok, [prompts[i] for i in rows.index], cand, batch_size=8)
    e, a, m = score_from_logprobs(lp)
    res = pd.concat([pd.DataFrame({'uid': rows.uid.values, 'split': split[rows.index].values, 'fold': k,
                                   'expected': e, 'argmax': a, 'valid_mass': m}), pd.DataFrame(lp, columns=lp_cols)], axis=1)
    hist.update(fold=k, n_train=len(tr), n_aug=len(aug), n_val=len(va), n_test=len(te), trainable_params=n_trainable, best=best,
                peak_vram_gb=torch.cuda.max_memory_allocated() / 1e9, fold_minutes=(time.time() - t_fold) / 60)
    json.dump(hist, open(f'{ROOT}/results/nb8_trainlog_f{k}.json', 'w'), indent=1, default=float)
    res.to_csv(out, index=False)
    shutil.rmtree(ck, ignore_errors=True)          # training state no longer needed
    print(f'fold {k} done; best epoch {best["epoch"]:.1f}; val Pearson {best["val_pearson"]:.3f}')
    del model; gc.collect(); torch.cuda.empty_cache()

==================== fold 0 ====================


config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

train 1535 + synthetic 307 {np.str_('negated'): 103, np.str_('keyword_salad'): 102, np.str_('swapped'): 102}  val 272  test 466
step 50/462  loss 0.7033  4.74s/step
step 100/462  loss 0.6993  4.62s/step
eval {'step': 116, 'epoch': 0.5021645021645021, 'val_nll': 1.3617879152297974, 'val_pearson': 0.7920872656897601, 'val_mae': 1.0788453174654364, 'val_mass': 0.9987723231315613}
checkpoint saved at step 116
step 150/462  loss 0.6617  5.10s/step
step 200/462  loss 0.6735  4.98s/step
eval {'step': 231, 'epoch': 1.0, 'val_nll': 0.9540252685546875, 'val_pearson': 0.7491978263231069, 'val_mae': 0.7677967902456393, 'val_mass': 0.9990108609199524}
checkpoint saved at step 231
step 250/462  loss 0.4802  5.17s/step
step 300/462  loss 0.4164  5.08s/step
eval {'step': 346, 'epoch': 1.4978354978354977, 'val_nll': 0.9361462593078613, 'val_pearson': 0.7762633349443432, 'val_mae': 0.6594937885908407, 'val_mass': 0.998992919921875}
checkpoint saved at step 346
step 350/462  loss 0.4738  5.20s/step
step 

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

224 4-bit layers; compute dtype before: ['torch.bfloat16'] -> torch.float16


/content/drive/MyDrive/asag_revision/perturb.py:68: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  good = good[~good.student_answer.astype(str).str.contains(NEG_WORDS)]


train 1550 + synthetic 310 {np.str_('negated'): 104, np.str_('swapped'): 103, np.str_('keyword_salad'): 103}  val 254  test 469
step 50/466  loss 0.8800  4.60s/step
step 100/466  loss 0.7754  4.61s/step
eval {'step': 116, 'epoch': 0.4978540772532189, 'val_nll': 1.6095638275146484, 'val_pearson': 0.7982554516941529, 'val_mae': 1.3685775097208277, 'val_mass': 0.994223415851593}
checkpoint saved at step 116
step 150/466  loss 0.6654  5.10s/step
step 200/466  loss 0.6858  4.99s/step
eval {'step': 233, 'epoch': 1.0, 'val_nll': 1.4739487171173096, 'val_pearson': 0.8057738835921501, 'val_mae': 1.2256615003189644, 'val_mass': 0.9971480965614319}
checkpoint saved at step 233
step 250/466  loss 0.4803  5.19s/step
step 300/466  loss 0.5100  5.11s/step
step 350/466  loss 0.3576  5.05s/step
eval {'step': 350, 'epoch': 1.502145922746781, 'val_nll': 1.679822325706482, 'val_pearson': 0.7639963786611479, 'val_mae': 1.0747875023704463, 'val_mass': 0.9989698529243469}
checkpoint saved at step 350
step 40

## 2. Results vs. the non-augmented model (runs once all folds are finished)

In [ ]:
done = [k for k in range(N_FOLDS) if os.path.exists(f'{ROOT}/results/mohler_{TAG}_f{k}.csv')]
if len(done) < N_FOLDS:
    print(f'{len(done)}/{N_FOLDS} folds finished. Re-run in a new session.')
else:
    from sklearn.linear_model import LinearRegression
    ft = pd.concat([pd.read_csv(f'{ROOT}/results/mohler_{TAG}_f{k}.csv') for k in range(N_FOLDS)])
    preds = pd.read_csv(f'{ROOT}/results/mohler_preds.csv')
    test = ft[ft.split == 'test'].set_index('uid')
    assert set(test.index) == set(preds.uid)
    cal = {}
    for k in range(N_FOLDS):
        v = ft[(ft.fold == k) & (ft.split == 'val')].merge(mo[['uid', 'y']], on='uid')
        lr = LinearRegression().fit(v[['expected']], v.y); t = test[test.fold == k]
        cal.update(dict(zip(t.index, np.clip(lr.predict(t[['expected']]), 0, 10))))
    preds[TAG] = preds.uid.map(test.expected); preds[f'{TAG}_cal'] = preds.uid.map(cal)
    preds.to_csv(f'{ROOT}/results/mohler_preds.csv', index=False)
    rows = [{'system': s, **summarise(preds.y, preds[s], reg_metrics, groups=preds.group, folds=preds.fold)}
            for s in ['llama3_ft', TAG, 'llama3_ft_cal', f'{TAG}_cal', 'llama3_instruct_zs']]
    summ = pd.DataFrame(rows); summ.to_csv(f'{ROOT}/results/mohler_nb8_summary.csv', index=False)
    display(summ[['system', 'pearson', 'pearson_ci', 'spearman', 'mae', 'rmse', 'qwk', 'pearson_fold_mean', 'pearson_fold_std']].round(3))
    # error on weak answers (gold < 6): does augmentation reduce over-scoring?
    low = preds[preds.y < 6]
    print('Gold < 6 (n=%d): mean signed error  ft %.2f  ftaug %.2f  instruct %.2f' %
          (len(low), (low.llama3_ft - low.y).mean(), (low[TAG] - low.y).mean(), (low.llama3_instruct_zs - low.y).mean()))

## 3. Training cost

In [ ]:
logs = [json.load(open(f'{ROOT}/results/nb8_trainlog_f{k}.json')) for k in range(N_FOLDS) if os.path.exists(f'{ROOT}/results/nb8_trainlog_f{k}.json')]
if logs:
    cost = pd.DataFrame([{'fold': h['fold'], 'train': h['n_train'], 'synthetic': h['n_aug'], 'steps': h['total_steps'],
                          'sec_per_step': h['sec_per_step'], 'peak_vram_gb': h['peak_vram_gb'],
                          'best_epoch': h['best']['epoch'], 'best_val_pearson': h['best']['val_pearson']} for h in logs])
    cost.to_csv(f'{ROOT}/results/nb8_training_cost.csv', index=False); display(cost.round(3))
json.dump(versions, open(f'{ROOT}/results/env_nb8.json', 'w'), indent=2)